In [ ]:
base_path_snapshots = None
Z_snapshot          = None
ecc                 = None
l_grid              = None
grid_chunks         = None
buffer_scale        = None
file_path_DTFE_Z    = None

In [ ]:
%run ./1___DTFE___Functions.ipynb

In [ ]:
ecc = np.array(eval(ecc))

---
---
---
---
---
---

In [ ]:
# the side in cMpc/h of each chunk
l_chunk  = np.diff(ecc[0])[0] / grid_chunks
core_vol = l_chunk**3

In [ ]:
# iterate over all chunks
for         ix in range(grid_chunks):
    for     iy in range(grid_chunks):
        for iz in range(grid_chunks):
            file_path_DTFE_Z_chunk = file_path_DTFE_Z+"chunk___"+str(ix)+"_"+str(iy)+"_"+str(iz)+"/"




            
            ### Get the coordinates in each chunk
            
            # Open the original data
            coords  = np.ascontiguousarray(illustris_load(base_path_snapshots+"/Original_Data",Z_snapshot,'dm',fields=['Coordinates']), dtype=np.float64)
            coords /= 1000   # kpc to Mpc

            # determine the minimum and maximum in the grid of the chunks' sides
            core_min = np.array([ix*l_chunk, iy*l_chunk, iz*l_chunk])
            core_max = core_min + np.array([l_chunk, l_chunk, l_chunk])

            # mask of all the coordinates inside the chunk
            mask_core = np.all((coords >= core_min) & (coords < core_max), axis=1)
            N_core = np.count_nonzero(mask_core)

            # the buffer size before scaling
            # it is the max of the average interparticle distance in this region fo the grid and the max of the average 
            #    interparticle distances at its edges
            buffer = np.max([(core_vol / N_core) ** (1/3), np.max(mean_edge_distances(coords[mask_core], core_min, core_max, frac=0.1))])
            buffer = min(buffer_scale * buffer, 0.15 * l_grid)
            with open(file_path_DTFE_Z_chunk+"buffer.pk", 'wb') as f: pkl.dump(buffer, f)

            # mirrored paddings/buffer
            # Shift the coordinates such that the minimum on each axis is set by the minimum of the buffered chunk and
            #    take the modulus of the total grid size.
            coords = np.mod(coords - core_min + buffer, l_grid)
            coords = coords[np.all(coords < l_chunk + 2*buffer, axis=1)]
            with open(file_path_DTFE_Z_chunk+"coords.pk", 'wb') as f: pkl.dump(coords, f)




            
            ### Compute the Delaunay tetrahedralization
            
            # DTFE first rquires we create a mesh of tetrahedrons: their coordinates and their 4 connections.
            connections = compute_tetgen_delaunay(coords)
            with open(file_path_DTFE_Z_chunk+"connections.pk", 'wb') as f: pkl.dump(connections, f)

            # Each tetrahedron contributes its mass to its 4 corners: each vertex density is mass / local volume around it.
            rho = compute_densities(coords, connections)
            with open(file_path_DTFE_Z_chunk+"rho.pk", 'wb') as f: pkl.dump(rho, f)

            # Later, when we compute the final grid, we reuqire the density gradients from our mesh.
            #D_rho = compute_gradients(coords, connections, rho)
            #with open(file_path_DTFE_Z_chunk+"D_rho.pk", 'wb') as f: pkl.dump(D_rho,  f)
            #del rho, D_rho; gc.collect()
            del rho; gc.collect()

            # When we later evaluate the density at an arbitrary position p (e.g. grid points for your final cube), we do not 
            #    want to check every tetrahedron (millions of them).
            # Instead, we query the tree for tetra centroids near p so we can only test those candidates for actual point-in-tetra
            #    containment (via barycentric coords).
            # Once we find the tetra that contains p, you use rho + Drho*r from that tetra to interpolate the density.
            # This step builds a search structure for fast interpolation. Without it, point location would be  O(N_tetra) per
            #    query — impossible at scale.

            # However, we use barycentric interpolation of vertex densities, not explicit gradient interpolation... so no more
            #    need for Drho at all.
            
            # the coordinates of the 4 vertices of each tetrahedron
            tetra_points = coords[connections]
            with open(file_path_DTFE_Z_chunk+"tetra_points.pk", 'wb') as f: pkl.dump(tetra_points,  f)
            del coords, connections; gc.collect()




            
            ### KDTree
            
            # This is a data structure used for fast nearest-neighbor searches in k-dimensional space.
            
            # For such a big set of 3D points (or in general anything higher than 2D), since we later need to find which points 
            #    are closest to a given query point, a tree structure makes this operation much faster!
            # Without a tree, for finding neighbors in N data points, one needs to check all N distances — that’s O(N) time 
            #    per query.A KDTree organizes the data so we can skip most of the points during the search and get results in 
            #    roughly O(log N) time per query.
            
            # geometric centroid of each tetrahedron
            centers = np.mean(tetra_points, axis=1)
            del tetra_points; gc.collect()

            tree = cKDTree(centers)
            with open(file_path_DTFE_Z_chunk+"tree.pk", 'wb') as f: pkl.dump(tree,  f)
            del tree, centers; gc.collect()

            ### Density reconstruction: precompute the inverses involved in the loop
            with open(file_path_DTFE_Z_chunk+"tetra_points.pk", 'rb') as f: tetra_points = pkl.load(f)
            Ainv_all, origins = precompute_inverses_fast(tetra_points)
            np.save(file_path_DTFE_Z_chunk+"Ainv_all.npy", Ainv_all)
            np.save(file_path_DTFE_Z_chunk+"origins.npy",  origins)
            del tetra_points, Ainv_all, origins; gc.collect()

---
---
---